In [5]:
# Part 4 – GenAI Integration
# Task 4.1 – Gemini API Integration and Structured Output
# Step1 – Install Required Libraries
!pip install -q google-genai pydantic pandas



In [6]:
# Step 2 – Import Required Libraries
import pandas as pd
import json
from pydantic import BaseModel, Field
from google import genai
from getpass import getpass

print("Libraries imported successfully!")


Libraries imported successfully!


In [7]:
# Step 3 - Load the dataset
notes_df = pd.read_csv('/content/followup_notes.csv')

print("Dataset loaded successfully!")
print("Dataset shape:", notes_df.shape)

display(notes_df.head(10))


Dataset loaded successfully!
Dataset shape: (30, 3)


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [8]:
# Step 4 – Connect Gemini API
api_key = getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=api_key)

print("Gemini API client created successfully!")


Enter your Gemini API Key: ··········
Gemini API client created successfully!


In [9]:
# Step 5 – Test the Gemini Flash Model
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain in one simple sentence why patients miss medical appointments."
)

print(response.text)


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [10]:

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain in one simple sentence why patients miss medical appointments."
)

print(response.text)


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [11]:

try:
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="Say hello in one sentence."
    )
    print(response.text)

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Full error:", str(e))


Error type: ServerError
Full error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [13]:

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello in one sentence."
)

print(response.text)


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [14]:
# Check available models
for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name, model.supported_actions)


models/gemini-2.5-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-4-26b-a4b-it ['generateContent', 'countTokens']
models/gemma-4-31b-it ['generateContent', 'countTokens']
models/gemini-flash-latest ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-flash-lite-latest ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-pro-latest ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-lite ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-image ['generateContent', 'countTokens', 'b

In [15]:

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Explain in one simple sentence why patients miss medical appointments."
)

print(response.text)


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash-lite is no longer available to new users. Please update your code to use models/gemini-3.5-flash-lite for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [16]:

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain in one simple sentence why patients miss medical appointments."
)

print(response.text)


Patients often miss medical appointments due to barriers like forgetting, lack of transportation, financial constraints, or scheduling conflicts.


In [17]:
# Step 6 - Define the Pydantic Output Structure
class NoteAnalysis(BaseModel):
    appointment_id: str
    summary: str
    root_cause: str
    recommended_action: str

print("Pydantic model created successfully!")


Pydantic model created successfully!


In [18]:
#Step 7 - Define allowed root-cause categories and JSON schema
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

MODEL_NAME = "gemini-3.5-flash-lite"

print("Allowed categories and JSON schema created successfully!")



Allowed categories and JSON schema created successfully!


In [19]:
#  Create the system instruction and CRAFT prompt
SYSTEM_TEXT = """
You are a healthcare appointment follow-up analysis assistant.
Analyse patient notes carefully.
Do not invent information or make unsupported assumptions.
Return only valid JSON matching the provided schema.
"""

def build_prompt(note_text):
    prompt = f"""
Context:
A clinic wants to understand why patients miss appointments.

Role:
You are an AI assistant analysing patient follow-up notes.

Action:
Read the note and identify the most likely root cause.
Choose only one of these categories:
{ALLOWED_ROOT_CAUSES}

If the reason is unclear, choose "other" instead of guessing.
Give a confidence score between 0 and 1.
Write a summary of at most 25 words.
Suggest one practical recommended action for the clinic.

Tone:
Be clear, professional and careful.

Format:
Return only JSON with these four fields:
root_cause, confidence, summary, recommended_action.

Patient note:
{note_text}
"""
    return prompt

print("CRAFT prompt created successfully!")


CRAFT prompt created successfully!


In [20]:
# Create analyse_note() Function
from google.genai import types

def analyse_note(note_text):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    data = json.loads(response.text)

    return data, response.text

print("analyse_note() function created successfully!")

analyse_note() function created successfully!


In [21]:
print("Available columns:", notes_df.columns.tolist())

first_note = notes_df.iloc[0]

print("\nFirst note:")
print(first_note)

Available columns: ['note_id', 'appointment_id', 'note_text']

First note:
note_id                                                        N001
appointment_id                                                A0872
note_text         Patient said she never got any reminder messag...
Name: 0, dtype: object


In [22]:
result, raw_response = analyse_note(first_note["note_text"])

print("Analysis result:")
print(result)

print("\nRaw JSON response:")
print(raw_response)

Analysis result:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'The patient missed the appointment because she did not receive a reminder and forgot, and she now wishes to reschedule.', 'recommended_action': "Verify and update the patient's contact information to ensure reminder notifications are delivered successfully."}

Raw JSON response:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.95,
  "summary": "The patient missed the appointment because she did not receive a reminder and forgot, and she now wishes to reschedule.",
  "recommended_action": "Verify and update the patient's contact information to ensure reminder notifications are delivered successfully."
}


In [23]:
try:
    result, raw_response = analyse_note(first_note["note_text"])

    print("Analysis result:")
    print(result)

    print("\nRaw JSON response:")
    print(raw_response)

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

Analysis result:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'The patient missed the appointment because she did not receive a reminder and forgot, and she wishes to reschedule.', 'recommended_action': "Verify and update the patient's contact information to ensure reliable automated appointment reminders."}

Raw JSON response:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.95,
  "summary": "The patient missed the appointment because she did not receive a reminder and forgot, and she wishes to reschedule.",
  "recommended_action": "Verify and update the patient's contact information to ensure reliable automated appointment reminders."
}


In [24]:
MODEL_NAME = "gemini-3.1-flash-lite"

print("Model changed to:", MODEL_NAME)

Model changed to: gemini-3.1-flash-lite


In [25]:
try:
    result, raw_response = analyse_note(first_note["note_text"])

    print("Analysis result:")
    print(result)

    print("\nRaw JSON response:")
    print(raw_response)

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

Analysis result:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient missed their appointment because they did not receive a reminder notification and subsequently forgot about the scheduled visit.', 'recommended_action': 'Implement an automated SMS or email reminder system sent 24-48 hours prior to all appointments.'}

Raw JSON response:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 1.0,
  "summary": "The patient missed their appointment because they did not receive a reminder notification and subsequently forgot about the scheduled visit.",
  "recommended_action": "Implement an automated SMS or email reminder system sent 24-48 hours prior to all appointments."
}


In [26]:
# Task 4.2 – Pydantic Validation and Processing Patient Notes
# 1 – Define Pydantic Validation Model
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

print("Pydantic validation model created successfully!")

Pydantic validation model created successfully!


In [27]:
# Step 2: Validate root-cause categories
def validate_insight(data):

    insight = FollowupInsight(**data)

    if insight.root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError(
            f"Invalid root cause: {insight.root_cause}"
        )

    return insight

print("Insight validation function created successfully!")

Insight validation function created successfully!


In [28]:
# Step 3: Process the first 10 patient notes
print("Total notes:", len(notes_df))
print("\nFirst 10 notes:")
display(notes_df.head(10))

print("\nMissing notes:", notes_df["note_text"].isna().sum())
print("Blank notes:", (notes_df["note_text"].fillna("").str.strip() == "").sum())

Total notes: 30

First 10 notes:


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...



Missing notes: 1
Blank notes: 1


In [29]:
# Step 4: Process and validate 10 notes
import time

results = []

for index, row in notes_df.head(10).iterrows():

    note_text = row["note_text"]

    # Prepare the result
    record = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": None,
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Skip blank notes
    if pd.isna(note_text) or str(note_text).strip() == "":

        record["root_cause"] = "other"
        record["confidence"] = 0.0
        record["summary"] = "No note recorded"
        record["recommended_action"] = "Re-call patient"
        record["validation_status"] = "skipped_blank"

        print(row["note_id"], "- Blank note skipped")

    else:
        try:
            data, raw_response = analyse_note(str(note_text))

            record["raw_response"] = raw_response

            # Copy the fields returned by Gemini
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:
                if field in data:
                    record[field] = data[field]

            # Validate the response
            validate_insight(data)

            record["validation_status"] = "valid"

            print(row["note_id"], "- Analysis successful")

        except Exception as e:

            if record["validation_status"] == "":
                record["validation_status"] = "invalid"

            print(row["note_id"], "- Error:", str(e))

        # Pause between API calls
        time.sleep(13)

    results.append(record)

print("\nProcessing completed!")
print("Total records processed:", len(results))

N001 - Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N002 - Analysis successful
N003 - Analysis successful
N004 - Blank note skipped
N005 - Analysis successful
N006 - Analysis successful
N007 - Analysis successful
N008 - Analysis successful
N009 - Analysis successful
N010 - Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Processing completed!
Total records processed: 10


In [32]:
# Step 5 – Retry only N001 and N010
for note_id in ["N001", "N010"]:

    row = notes_df[notes_df["note_id"] == note_id].iloc[0]

    print("Retrying:", note_id)

    try:
        data, raw_response = analyse_note(row["note_text"])

        validate_insight(data)

        # Update the existing record
        for record in results:
            if record["note_id"] == note_id:

                record["root_cause"] = data["root_cause"]
                record["confidence"] = data["confidence"]
                record["summary"] = data["summary"]
                record["recommended_action"] = data["recommended_action"]
                record["validation_status"] = "valid"
                record["raw_response"] = raw_response

        print(note_id, "- Retry successful!")

    except Exception as e:
        print(note_id, "- Retry failed:", str(e))

    time.sleep(13)

print("\nRetry process completed!")

Retrying: N001
N001 - Retry successful!
Retrying: N010
N010 - Retry successful!

Retry process completed!


In [33]:
# Step 6 – Save the analysed results
results_df = pd.DataFrame(results)

# Arrange columns in the required order
results_df = results_df[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

# Save the file
results_df.to_csv("notes_analysed.csv", index=False)

print("File saved successfully!")
print("File shape:", results_df.shape)

print("\nValidation status:")
print(results_df["validation_status"].value_counts())

print("\nFirst 10 analysed records:")
display(results_df)

File saved successfully!
File shape: (10, 8)

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

First 10 analysed records:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,1.0,The patient missed their appointment because t...,Implement an automated SMS or email reminder s...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ..."
1,N002,A0897,distance_or_transport,1.0,The patient missed their appointment due to th...,Offer telehealth consultations for routine che...,valid,"{\n ""root_cause"": ""distance_or_transport"",\n ..."
2,N003,A0526,cost_or_fee,1.0,The patient missed their appointment due to th...,Implement a sliding scale fee structure or off...,valid,"{\n ""root_cause"": ""cost_or_fee"",\n ""confiden..."
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,1.0,The patient missed their appointment due to a ...,Send a follow-up message to the patient once a...,valid,"{\n ""root_cause"": ""illness_or_emergency"",\n ..."
5,N006,A0507,work_or_schedule,1.0,The patient missed their appointment due to an...,Offer extended clinic hours or evening appoint...,valid,"{\n ""root_cause"": ""work_or_schedule"",\n ""con..."
6,N007,A0333,forgot_or_no_reminder,1.0,The patient missed the appointment because the...,Implement an automated SMS reminder system sen...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ..."
7,N008,A0222,distance_or_transport,1.0,The patient missed the appointment due to a ci...,Offer telehealth consultations for patients re...,valid,"{\n ""root_cause"": ""distance_or_transport"",\n ..."
8,N009,A0977,other,0.9,The clinic attempted to contact the patient th...,Implement an automated SMS reminder system to ...,valid,"{\n ""root_cause"": ""other"",\n ""confidence"": 0..."
9,N010,A0596,work_or_schedule,1.0,The patient missed the appointment due to a ma...,Offer the patient Saturday appointment slots t...,valid,"{\n ""root_cause"": ""work_or_schedule"",\n ""con..."


In [34]:
from google.colab import files
files.download("notes_analysed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
# Task 4.3 – Break It on Purpose.
# Step 1 – Test invalid AI responses
bad_response = {
    "root_cause": "unknown_reason",
    "confidence": 1.5,
    "summary": "Bad",
    "recommended_action": "No"
}

print("Test response created:")
print(bad_response)

Test response created:
{'root_cause': 'unknown_reason', 'confidence': 1.5, 'summary': 'Bad', 'recommended_action': 'No'}


In [36]:
# Step 2: Test the validation system
try:
    validate_insight(bad_response)

    print("Validation passed!")

except Exception as e:
    print("Validation failed successfully!")
    print("\nDetected errors:")
    print(e)

Validation failed successfully!

Detected errors:
3 validation errors for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
summary
  String should have at least 5 characters [type=string_too_short, input_value='Bad', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
recommended_action
  String should have at least 5 characters [type=string_too_short, input_value='No', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short


In [37]:
# Step 3: Test invalid root cause
bad_root_response = {
    "root_cause": "unknown_reason",
    "confidence": 0.8,
    "summary": "Patient missed the appointment.",
    "recommended_action": "Call the patient again."
}

try:
    validate_insight(bad_root_response)

    print("Validation passed!")

except Exception as e:
    print("Invalid root cause detected!")
    print(e)

Invalid root cause detected!
Invalid root cause: unknown_reason
